# LibBOT — BM25 vs MiniLM (prefix / windowed) vs hybrid retrieval on SciFact

This is the **one shared notebook** for the whole pipeline. Each member owns one section. Sections talk to each other **only through files on disk** (see the *Data contract* below), so you can start from your own section without re-running everyone else's.

| § | Section | Owner | Status |
|---|---|---|---|
| 1 | Setup, config, shared helpers, data contract | Member 1 (scaffolded by M2) | ✅ implemented |
| 2 | Dataset loading + claim→document→evidence mapping | **Member 2** | ✅ implemented |
| A | Checkpoint A: load shared data | shared | ✅ implemented |
| 3 | BM25 baseline | Member 3 | ⬜ scaffold |
| 4 | Prefix MiniLM | Member 4 | ⬜ scaffold |
| 5 | Windowed MiniLM | Member 5 | ⬜ scaffold |
| 6 | Truncation / evidence-retention diagnostic | **Members 2 + 5** | 🟨 6.1 implemented (M2), rest scaffold |
| 7 | Hybrid fusion + α tuning | Member 6 | ⬜ scaffold |
| 8 | Final evaluation (quality + efficiency) | Member 1 | ⬜ scaffold |
| 9 | Statistical comparison, results table, charts | Member 6 | ⬜ scaffold |
| 10 | Demo export + reproducibility log | Member 6 / Member 1 | ⬜ scaffold |

## Working rules (read once)

1. **Edit only your own section.** Changes to Section 1 (config/helpers) need a heads-up to the group, because everyone depends on them.
2. **Inputs and outputs go through disk.** Read from `data/processed/` and write your rankings with `save_run()` into `results/runs/`. Never pass data between sections via a variable that only exists in your own session.
3. **One ranking format for everyone.** Use `scores_to_run()` + `save_run()`. `save_run()` validates the format and refuses bad files.
4. **No test leakage.** α tuning, window settings and any other choice uses **train** only. Test qrels and evidence locations are only for final evaluation and the diagnostic.
5. **Git hygiene for a shared notebook:** run `nbstripout --install` once after cloning (strips outputs on commit), always `git pull` before you start, and commit only changes inside your section. If two people must edit at once, work on branches and merge one at a time.
6. **Run order:** Section 1 → (Section 2 *or* Checkpoint A) → your section.

---
# 1 · Setup, config and shared helpers  *(Member 1 — scaffolded by Member 2)*

In [ ]:
# Uncomment on a fresh machine / Colab.
# On Colab, first:  !git clone https://github.com/yapzukai/libbot.git  then  %cd libbot
# %pip install -q -r requirements.txt

In [ ]:
import os, sys, re, json, time, random, platform, hashlib, tarfile, zipfile, shutil, urllib.request, warnings
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 160)

# ---------------- Global config (team-wide; change only by agreement) ----------------
SEED        = 42
K           = 10        # evaluation cutoff: Recall@10, nDCG@10
RUN_DEPTH   = 1000      # docs saved per query in every run file (fusion needs more than the top-10)
MODEL_NAME  = "sentence-transformers/all-MiniLM-L6-v2"
MAX_SEQ_LEN = 256       # MiniLM input limit in word-pieces, INCLUDING [CLS] and [SEP]
DOC_SEP     = " "       # doc_text = (title + DOC_SEP + text).strip()  -> the ONE string every method encodes/indexes
ALPHA_GRID  = [round(a, 2) for a in np.linspace(0.0, 1.0, 11)]   # same grid for both hybrids
METHODS     = ["bm25", "prefix_minilm", "window_minilm", "hybrid_prefix", "hybrid_window"]
SPLITS      = ["train", "test"]   # BEIR splits. train = tuning, test = final (300 queries)

# ---------------- Paths ----------------
ROOT        = Path(os.environ.get("LIBBOT_ROOT", Path.cwd())).resolve()
RAW_DIR     = ROOT / "data" / "raw"
PROC_DIR    = ROOT / "data" / "processed"
CACHE_DIR   = ROOT / "cache"
RESULTS_DIR = ROOT / "results"
RUNS_DIR    = RESULTS_DIR / "runs"
METRICS_DIR = RESULTS_DIR / "metrics"
EFF_DIR     = RESULTS_DIR / "efficiency"
FIG_DIR     = RESULTS_DIR / "figures"
for d in [RAW_DIR, PROC_DIR, CACHE_DIR, RUNS_DIR, METRICS_DIR, EFF_DIR, FIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)

def environment_info():
    info = {
        "timestamp_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "python": sys.version.split()[0],
        "platform": platform.platform(),
        "processor": platform.processor() or platform.machine(),
        "cpu_count": os.cpu_count(),
        "numpy": np.__version__, "pandas": pd.__version__,
    }
    try:
        import torch
        info["torch"] = torch.__version__
        info["cuda"] = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
        info["mps"] = bool(getattr(torch.backends, "mps", None) and torch.backends.mps.is_available())
    except ImportError:
        info["torch"] = None
    for lib in ["transformers", "sentence_transformers", "rank_bm25"]:
        try:
            info[lib] = __import__(lib).__version__
        except Exception:
            info[lib] = None
    return info

ENV = environment_info()
CONFIG = dict(SEED=SEED, K=K, RUN_DEPTH=RUN_DEPTH, MODEL_NAME=MODEL_NAME, MAX_SEQ_LEN=MAX_SEQ_LEN,
              DOC_SEP=DOC_SEP, ALPHA_GRID=ALPHA_GRID, METHODS=METHODS, SPLITS=SPLITS)
(RESULTS_DIR / "config_snapshot.json").write_text(json.dumps({"config": CONFIG, "environment": ENV}, indent=2))
print("ROOT:", ROOT)
pd.Series(ENV)

### Data contract (the shared format)

All IDs are **strings** (`doc_id`, `query_id`). `doc_text` is the single canonical document string; character offsets below are offsets into `doc_text`.

**`data/processed/`** — written by Section 2

| File | One row per | Key columns |
|---|---|---|
| `corpus.parquet` | document (5,183) | `doc_id, title, text, doc_text, sentences[list], sent_start[list], sent_end[list], n_sentences, n_words, structured` |
| `queries.parquet` | claim (1,109) | `query_id, claim, split ∈ {train,test,unused}, orig_split ∈ {train,dev}` |
| `qrels.parquet` | (query, relevant doc) | `query_id, doc_id, relevance, split` |
| `qrel_pairs.parquet` | (query, relevant doc) + evidence | qrels columns + `claim, has_evidence, label, evidence_sentence_ids[list], ev_char_start[list], ev_char_end[list], first_evidence_char, rel_position_first, …` |
| `evidence_sentences.parquet` | evidence sentence | `query_id, doc_id, split, label, rationale_set, sentence_idx, char_start, char_end, sentence_text` |
| `manifest.json` | — | counts, file hashes, data source, timestamp |

Written by Section 6.1: `doc_token_stats.parquet` (per doc: word-piece count, retained character span) and `prefix_retention.parquet` (per qrel pair: complete / partial / absent).

**`results/runs/<method>__<split>.parquet`** — one per method × split, written with `save_run()`

| column | type | meaning |
|---|---|---|
| `query_id` | str | every evaluation query of that split |
| `doc_id` | str | must exist in `corpus.parquet` |
| `score` | float | higher = more relevant (raw score; fusion normalises later) |
| `rank` | int | 1 … `RUN_DEPTH`, contiguous, sorted by score desc (ties → doc_id asc) |

`<method>` ∈ `bm25, prefix_minilm, window_minilm, hybrid_prefix, hybrid_window`.

**`results/efficiency/efficiency_log.jsonl`** — append with `log_efficiency(method, ...)`: `indexing_time_s, n_embeddings, storage_bytes, mean_latency_ms, p95_latency_ms, ...` plus hardware info automatically.

In [ ]:
# ---------------- Shared helpers (format + validation). Everyone uses these. ----------------

def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

def write_jsonl(rows, path):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

def norm_ws(s):
    """Collapse all whitespace runs to one space (for robust text comparison)."""
    return re.sub(r"\s+", " ", str(s)).strip()

def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

CHECK_LOG = []
def check(name, ok, detail="", hard=False):
    """Record a sanity check. hard=True raises on failure; otherwise it is a warning."""
    ok = bool(ok)
    status = "PASS" if ok else ("FAIL" if hard else "WARN")
    CHECK_LOG.append({"check": name, "status": status, "detail": str(detail)})
    print(f"[{status}] {name}" + (f"  — {detail}" if detail else ""))
    if hard and not ok:
        raise AssertionError(f"{name}: {detail}")
    return ok

# ---- processed data registry ----
PROCESSED_FILES = {
    "corpus":             PROC_DIR / "corpus.parquet",
    "queries":            PROC_DIR / "queries.parquet",
    "qrels":              PROC_DIR / "qrels.parquet",
    "qrel_pairs":         PROC_DIR / "qrel_pairs.parquet",
    "evidence_sentences": PROC_DIR / "evidence_sentences.parquet",
    "doc_token_stats":    PROC_DIR / "doc_token_stats.parquet",
    "prefix_retention":   PROC_DIR / "prefix_retention.parquet",
}

def _restore_lists(df):
    # parquet round-trips list columns as numpy arrays; turn them back into python lists
    for c in df.columns:
        if len(df) and isinstance(df[c].iloc[0], np.ndarray):
            df[c] = df[c].apply(list)
    return df

def load_processed(*names):
    """load_processed('corpus', 'qrels') -> tuple of DataFrames (single DataFrame if one name)."""
    out = []
    for n in names:
        p = PROCESSED_FILES[n]
        if not p.exists():
            raise FileNotFoundError(f"{p} not found — run Section 2 (or pull data/processed/ from the repo).")
        out.append(_restore_lists(pd.read_parquet(p)))
    return out[0] if len(out) == 1 else tuple(out)

# ---- evaluation-query helpers ----
def eval_queries(split, queries_df=None):
    """Queries that have at least one qrel in `split` (BEIR convention: only these are evaluated)."""
    q = queries_df if queries_df is not None else load_processed("queries")
    return q[q["split"] == split].reset_index(drop=True)

def qrels_dict(split, qrels_df=None):
    """{query_id: {doc_id: relevance}} — the format pytrec_eval / BEIR evaluators expect."""
    qr = qrels_df if qrels_df is not None else load_processed("qrels")
    qr = qr[qr["split"] == split]
    out = {}
    for qid, did, rel in qr[["query_id", "doc_id", "relevance"]].itertuples(index=False):
        out.setdefault(qid, {})[did] = int(rel)
    return out

# ---- run (ranking) format ----
RUN_COLUMNS = ["query_id", "doc_id", "score", "rank"]

def run_path(method, split):
    return RUNS_DIR / f"{method}__{split}.parquet"

def scores_to_run(scores, query_ids, doc_ids, depth=RUN_DEPTH):
    """Turn a (n_queries x n_docs) score matrix (higher = better) into the shared run format."""
    scores = np.asarray(scores, dtype=np.float64)
    assert scores.shape == (len(query_ids), len(doc_ids)), f"scores {scores.shape} vs ({len(query_ids)}, {len(doc_ids)})"
    depth = min(depth, scores.shape[1])
    doc_ids = np.asarray([str(d) for d in doc_ids], dtype=object)
    parts = []
    for i, qid in enumerate(query_ids):
        s = scores[i]
        top = np.argpartition(-s, depth - 1)[:depth] if depth < len(s) else np.arange(len(s))
        parts.append(pd.DataFrame({"query_id": str(qid), "doc_id": doc_ids[top], "score": s[top]}))
    run = pd.concat(parts, ignore_index=True)
    return _rank_run(run)

def _rank_run(run):
    run = run.sort_values(["query_id", "score", "doc_id"], ascending=[True, False, True], kind="mergesort")
    run["rank"] = run.groupby("query_id").cumcount() + 1
    return run.reset_index(drop=True)[RUN_COLUMNS]

_CORPUS_IDS = None
def _corpus_ids():
    global _CORPUS_IDS
    if _CORPUS_IDS is None and PROCESSED_FILES["corpus"].exists():
        _CORPUS_IDS = set(pd.read_parquet(PROCESSED_FILES["corpus"], columns=["doc_id"])["doc_id"])
    return _CORPUS_IDS

def validate_run(run, split=None, depth=RUN_DEPTH, name="run"):
    """Raises AssertionError if `run` breaks the shared format."""
    missing = [c for c in RUN_COLUMNS if c not in run.columns]
    assert not missing, f"{name}: missing columns {missing}"
    assert run["query_id"].map(type).eq(str).all() and run["doc_id"].map(type).eq(str).all(), f"{name}: ids must be str"
    assert np.isfinite(run["score"].to_numpy(dtype=float)).all(), f"{name}: non-finite scores"
    assert not run.duplicated(["query_id", "doc_id"]).any(), f"{name}: duplicate (query_id, doc_id) rows"
    g = run.sort_values(["query_id", "rank"]).groupby("query_id")
    sizes = g.size()
    assert (sizes <= depth).all(), f"{name}: more than {depth} docs for some query"
    ranks_ok = g["rank"].apply(lambda r: (r.to_numpy() == np.arange(1, len(r) + 1)).all())
    assert ranks_ok.all(), f"{name}: ranks must be 1..n per query; bad queries e.g. {list(ranks_ok[~ranks_ok].index[:3])}"
    mono = g["score"].apply(lambda s: bool((np.diff(s.to_numpy()) <= 1e-12).all()))
    assert mono.all(), f"{name}: scores must be non-increasing with rank"
    cids = _corpus_ids()
    if cids is not None:
        bad = set(run["doc_id"]) - cids
        assert not bad, f"{name}: {len(bad)} doc_ids not in corpus, e.g. {list(bad)[:3]}"
    if split is not None and PROCESSED_FILES["queries"].exists():
        expected = set(eval_queries(split)["query_id"])
        got = set(run["query_id"])
        assert got == expected, f"{name}: query set mismatch for split={split} (missing {len(expected-got)}, extra {len(got-expected)})"
    return True

def save_run(run, method, split, depth=RUN_DEPTH):
    assert method in METHODS, f"method must be one of {METHODS}"
    assert split in SPLITS, f"split must be one of {SPLITS}"
    run = run[RUN_COLUMNS].copy()
    run["query_id"] = run["query_id"].astype(str)
    run["doc_id"] = run["doc_id"].astype(str)
    run["rank"] = run["rank"].astype(int)
    validate_run(run, split=split, depth=depth, name=f"{method}/{split}")
    path = run_path(method, split)
    run.to_parquet(path, index=False)
    print(f"saved {path.relative_to(ROOT)}  ({run['query_id'].nunique()} queries, {len(run):,} rows)")
    return path

def load_run(method, split):
    return pd.read_parquet(run_path(method, split))

def run_status():
    rows = []
    for m in METHODS:
        for s in SPLITS:
            p = run_path(m, s)
            rows.append({"method": m, "split": s, "exists": p.exists(),
                         "rows": len(pd.read_parquet(p, columns=["rank"])) if p.exists() else 0})
    return pd.DataFrame(rows).pivot(index="method", columns="split", values="exists").reindex(METHODS)

def log_efficiency(method, **metrics):
    """Append one efficiency record, e.g. log_efficiency('bm25', indexing_time_s=1.2, mean_latency_ms=9.0)."""
    rec = {"method": method, "timestamp_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
           "hardware": f"{ENV['processor']} | cuda={ENV.get('cuda')} | mps={ENV.get('mps')}", **metrics}
    with open(EFF_DIR / "efficiency_log.jsonl", "a", encoding="utf-8") as f:
        f.write(json.dumps(rec) + "\n")
    return rec

print("helpers ready")

---
# 2 · Dataset loading and evidence mapping  *(Member 2)*

**Goal.** Load both versions of SciFact and join them into one clean set of tables:

- **BEIR SciFact** gives the retrieval task: corpus, queries (claims) and relevance labels (qrels, train/test).
- **Original SciFact** gives what BEIR drops: the abstract split into sentences, and for every claim–document pair the **expert-annotated evidence sentences** and label (SUPPORT / CONTRADICT).

The output is a mapping **claim → relevant document → relevance label → evidence sentences (with character positions inside `doc_text`)**. Section 6 uses those positions to check whether evidence falls past MiniLM's 256-word-piece cutoff.

**One split detail matters a lot.** BEIR's *test* qrels (300 claims) are built from the original SciFact ***dev*** split, and BEIR *train* comes from original *train*. The original *test* split has no public labels. Section 2.3 checks this mapping from the data instead of assuming it.

| Step | What |
|---|---|
| 2.1 | Download raw files (direct links, with a Hugging Face fallback) |
| 2.2 | Load raw BEIR + original files |
| 2.3 | Alignment checks: corpus, claims, split mapping |
| 2.4 | Canonical corpus table with sentence character spans |
| 2.5 | Queries + qrels tables |
| 2.6 | Evidence mapping (qrel pairs ↔ rationale sentences) |
| 2.7 | Mapping validation (*Fair evaluation protocol, step 1*) |
| 2.8 | Descriptive stats + figures |
| 2.9 | Save processed files + manifest |

### 2.1 Download raw files

In [ ]:
BEIR_URL = "https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/scifact.zip"
ORIG_URL = "https://scifact.s3-us-west-2.amazonaws.com/release/latest/data.tar.gz"

BEIR_DIR = RAW_DIR / "beir" / "scifact"
ORIG_DIR = RAW_DIR / "scifact_original" / "data"
BEIR_FILES = [BEIR_DIR / "corpus.jsonl", BEIR_DIR / "queries.jsonl",
              BEIR_DIR / "qrels" / "train.tsv", BEIR_DIR / "qrels" / "test.tsv"]
ORIG_FILES = [ORIG_DIR / f for f in ["corpus.jsonl", "claims_train.jsonl", "claims_dev.jsonl", "claims_test.jsonl"]]
DATA_SOURCE = {}

def _download(url, dest):
    dest.parent.mkdir(parents=True, exist_ok=True)
    print(f"downloading {url}")
    req = urllib.request.Request(url, headers={"User-Agent": "libbot-is424"})
    with urllib.request.urlopen(req, timeout=120) as r, open(dest, "wb") as f:
        shutil.copyfileobj(r, f)
    return dest

def _safe_extract_tar(tar_path, out_dir):
    with tarfile.open(tar_path) as tf:
        for m in tf.getmembers():
            target = (out_dir / m.name).resolve()
            assert str(target).startswith(str(out_dir.resolve())), f"unsafe path in archive: {m.name}"
        tf.extractall(out_dir)

def _beir_from_hf():
    """Fallback: rebuild the BEIR raw layout from the Hugging Face copies."""
    from datasets import load_dataset
    corpus = load_dataset("BeIR/scifact", "corpus", split="corpus")
    queries = load_dataset("BeIR/scifact", "queries", split="queries")
    qrels = load_dataset("BeIR/scifact-qrels")
    write_jsonl([{"_id": str(r["_id"]), "title": r["title"], "text": r["text"]} for r in corpus], BEIR_DIR / "corpus.jsonl")
    write_jsonl([{"_id": str(r["_id"]), "text": r["text"]} for r in queries], BEIR_DIR / "queries.jsonl")
    for s in SPLITS:
        df = qrels[s].to_pandas()
        (BEIR_DIR / "qrels").mkdir(parents=True, exist_ok=True)
        df.columns = ["query-id", "corpus-id", "score"]
        df.to_csv(BEIR_DIR / "qrels" / f"{s}.tsv", sep="\t", index=False)

def _original_from_hf():
    """Fallback: rebuild the original raw layout from Hugging Face (one row per evidence set there)."""
    from datasets import load_dataset
    rev = "refs/convert/parquet"   # allenai/scifact is a script dataset; this revision has plain parquet
    corpus = load_dataset("allenai/scifact", "corpus", revision=rev, split="train")
    write_jsonl([{"doc_id": int(r["doc_id"]), "title": r["title"], "abstract": list(r["abstract"]),
                  "structured": bool(r["structured"])} for r in corpus], ORIG_DIR / "corpus.jsonl")
    for hf_split, name in [("train", "train"), ("validation", "dev"), ("test", "test")]:
        claims = {}
        for r in load_dataset("allenai/scifact", "claims", revision=rev, split=hf_split):
            c = claims.setdefault(int(r["id"]), {"id": int(r["id"]), "claim": r["claim"], "evidence": {},
                                                 "cited_doc_ids": list(r.get("cited_doc_ids") or [])})
            if r.get("evidence_doc_id"):
                c["evidence"].setdefault(str(r["evidence_doc_id"]), []).append(
                    {"sentences": list(r["evidence_sentences"]), "label": r["evidence_label"]})
        write_jsonl(list(claims.values()), ORIG_DIR / f"claims_{name}.jsonl")

def fetch_beir():
    if all(p.exists() for p in BEIR_FILES):
        return "cached"
    try:
        z = _download(BEIR_URL, RAW_DIR / "beir" / "scifact.zip")
        with zipfile.ZipFile(z) as zf:
            zf.extractall(RAW_DIR / "beir")
        return BEIR_URL
    except Exception as e:
        print(f"  direct download failed ({e!r}); trying Hugging Face …")
        _beir_from_hf()
        return "huggingface:BeIR/scifact + BeIR/scifact-qrels"

def fetch_original():
    if all(p.exists() for p in ORIG_FILES):
        return "cached"
    try:
        t = _download(ORIG_URL, RAW_DIR / "scifact_original" / "data.tar.gz")
        _safe_extract_tar(t, RAW_DIR / "scifact_original")
        return ORIG_URL
    except Exception as e:
        print(f"  direct download failed ({e!r}); trying Hugging Face …")
        _original_from_hf()
        return "huggingface:allenai/scifact (parquet)"

DATA_SOURCE["beir"] = fetch_beir()
DATA_SOURCE["original"] = fetch_original()
for p in BEIR_FILES + ORIG_FILES:
    check(f"raw file present: {p.relative_to(ROOT)}", p.exists(), hard=True)
DATA_SOURCE

> **If both download routes fail** (e.g. a firewall), download manually and place the files so these paths exist:
> `data/raw/beir/scifact/{corpus.jsonl, queries.jsonl, qrels/train.tsv, qrels/test.tsv}` from the BEIR zip, and
> `data/raw/scifact_original/data/{corpus.jsonl, claims_train.jsonl, claims_dev.jsonl, claims_test.jsonl}` from the SciFact `data.tar.gz`. Then re-run.

### 2.2 Load raw files

In [ ]:
# ---- BEIR ----
beir_corpus = (pd.DataFrame(read_jsonl(BEIR_DIR / "corpus.jsonl"))
               .rename(columns={"_id": "doc_id"})[["doc_id", "title", "text"]])
beir_corpus["doc_id"] = beir_corpus["doc_id"].astype(str)
beir_corpus[["title", "text"]] = beir_corpus[["title", "text"]].fillna("")

beir_queries = (pd.DataFrame(read_jsonl(BEIR_DIR / "queries.jsonl"))
                .rename(columns={"_id": "query_id", "text": "claim"})[["query_id", "claim"]])
beir_queries["query_id"] = beir_queries["query_id"].astype(str)

beir_qrels = []
for s in SPLITS:
    q = pd.read_csv(BEIR_DIR / "qrels" / f"{s}.tsv", sep="\t", dtype=str)
    q.columns = ["query_id", "doc_id", "relevance"]
    q["relevance"] = q["relevance"].astype(int)
    q["split"] = s
    beir_qrels.append(q)
beir_qrels = pd.concat(beir_qrels, ignore_index=True)

# ---- Original SciFact ----
orig_corpus = {str(d["doc_id"]): d for d in read_jsonl(ORIG_DIR / "corpus.jsonl")}
ORIG_SPLITS = ["train", "dev", "test"]
orig_claims = {s: {str(c["id"]): c for c in read_jsonl(ORIG_DIR / f"claims_{s}.jsonl")} for s in ORIG_SPLITS}

raw_counts = pd.Series({
    "BEIR corpus docs": len(beir_corpus),
    "BEIR queries": len(beir_queries),
    "BEIR train qrel rows": int((beir_qrels.split == "train").sum()),
    "BEIR train queries w/ qrels": beir_qrels.loc[beir_qrels.split == "train", "query_id"].nunique(),
    "BEIR test qrel rows": int((beir_qrels.split == "test").sum()),
    "BEIR test queries w/ qrels": beir_qrels.loc[beir_qrels.split == "test", "query_id"].nunique(),
    "Original corpus docs": len(orig_corpus),
    **{f"Original {s} claims": len(orig_claims[s]) for s in ORIG_SPLITS},
    **{f"Original {s} claims with evidence": sum(bool(c.get("evidence")) for c in orig_claims[s].values()) for s in ORIG_SPLITS},
}, name="count")
raw_counts.to_frame()

### 2.3 Alignment checks

Expected sizes (from the SciFact paper and BEIR): corpus 5,183 · BEIR queries 1,109 · BEIR test queries 300 · original claims 809 train / 300 dev / 300 test.
*(The proposal slide's 1,261 / 450 / 300 counts rows in the flattened Hugging Face version, with one row per evidence set, so claims with several evidence documents appear more than once. Counted per claim it is 809 / 300 / 300.)*

Count mismatches are **warnings** (useful if a dataset version changes). Broken joins are **hard failures**.

In [ ]:
EXPECTED = {"corpus": 5183, "beir_queries": 1109, "beir_test_queries": 300,
            "orig_train": 809, "orig_dev": 300, "orig_test": 300}
check("BEIR corpus size", len(beir_corpus) == EXPECTED["corpus"], f"{len(beir_corpus)} vs {EXPECTED['corpus']}")
check("Original corpus size", len(orig_corpus) == EXPECTED["corpus"], f"{len(orig_corpus)} vs {EXPECTED['corpus']}")
check("BEIR queries", len(beir_queries) == EXPECTED["beir_queries"], f"{len(beir_queries)} vs {EXPECTED['beir_queries']}")
n_test_q = beir_qrels.loc[beir_qrels.split == "test", "query_id"].nunique()
check("BEIR test queries with qrels", n_test_q == EXPECTED["beir_test_queries"], f"{n_test_q} vs {EXPECTED['beir_test_queries']}")
for s in ORIG_SPLITS:
    check(f"Original {s} claims", len(orig_claims[s]) == EXPECTED[f"orig_{s}"], f"{len(orig_claims[s])} vs {EXPECTED[f'orig_{s}']}")

# ---- corpus: same documents, same text? ----
beir_ids, orig_ids = set(beir_corpus.doc_id), set(orig_corpus)
check("corpus doc_id sets identical (BEIR vs original)", beir_ids == orig_ids,
      f"only-BEIR={len(beir_ids - orig_ids)}, only-original={len(orig_ids - beir_ids)}", hard=False)
common = beir_corpus[beir_corpus.doc_id.isin(orig_ids)]
title_match = np.mean([norm_ws(t) == norm_ws(orig_corpus[d]["title"]) for d, t in zip(common.doc_id, common.title)])
text_match = np.mean([norm_ws(x) == norm_ws(" ".join(orig_corpus[d]["abstract"])) for d, x in zip(common.doc_id, common.text)])
check("BEIR title == original title (whitespace-normalised)", title_match == 1.0, f"{title_match:.2%} match")
check("BEIR text == ' '.join(original abstract sentences)", text_match == 1.0, f"{text_match:.2%} match")

# ---- claims: BEIR query ids/text match original claims ----
all_orig = {qid: (s, c) for s in ORIG_SPLITS for qid, c in orig_claims[s].items()}
in_orig = beir_queries.query_id.isin(all_orig)
check("every BEIR query_id exists in original claims", in_orig.all(), f"{(~in_orig).sum()} missing", hard=True)
claim_match = np.mean([norm_ws(c) == norm_ws(all_orig[q][1]["claim"]) for q, c in zip(beir_queries.query_id, beir_queries.claim)])
check("BEIR query text == original claim text", claim_match == 1.0, f"{claim_match:.2%} match")

# ---- split mapping, derived from the data ----
overlap = pd.DataFrame({os_: {bs: beir_qrels.loc[beir_qrels.split == bs, "query_id"].drop_duplicates().isin(orig_claims[os_]).sum()
                             for bs in SPLITS} for os_ in ORIG_SPLITS})
overlap.index.name = "BEIR split ↓ / original split →"
display(overlap)
SPLIT_MAP = {bs: overlap.loc[bs].idxmax() for bs in SPLITS}
for bs, os_ in SPLIT_MAP.items():
    n = beir_qrels.loc[beir_qrels.split == bs, "query_id"].nunique()
    check(f"BEIR {bs} queries all come from original {os_}", overlap.loc[bs, os_] == n,
          f"{overlap.loc[bs, os_]}/{n}", hard=True)
check("BEIR test ↔ original dev (expected)", SPLIT_MAP.get("test") == "dev", f"SPLIT_MAP={SPLIT_MAP}")
print("SPLIT_MAP =", SPLIT_MAP)

### 2.4 Canonical corpus table with sentence character spans

`doc_text = (title + " " + text).strip()` is **the** string every method indexes or encodes (it matches the BEIR / Sentence-Transformers convention). Each original abstract sentence is located inside `doc_text`. An exact match is tried first; if that fails, a whitespace-tolerant match is used. This gives `sent_start[i]`, `sent_end[i]` for sentence `i`.

In [ ]:
def build_doc_text(title, text, sep=DOC_SEP):
    return (f"{title}{sep}{text}" if title else text).strip()

def locate_sentences(doc_text, sentences, start=0):
    """Return (starts, ends) char offsets of each sentence inside doc_text, searching left-to-right.
    -1 means the sentence could not be located."""
    starts, ends, cursor = [], [], start
    for sent in sentences:
        s = str(sent).strip()
        if not s:
            starts.append(-1); ends.append(-1); continue
        i = doc_text.find(s, cursor)
        if i >= 0:
            j = i + len(s)
        else:
            m = re.compile(r"\s+".join(map(re.escape, s.split()))).search(doc_text, cursor)
            if not m:
                starts.append(-1); ends.append(-1); continue
            i, j = m.start(), m.end()
        starts.append(i); ends.append(j); cursor = j
    return starts, ends

rows = []
for d in beir_corpus.itertuples(index=False):
    o = orig_corpus.get(d.doc_id, {})
    sents = [str(s) for s in o.get("abstract", [])]
    doc_text = build_doc_text(d.title, d.text)
    body_offset = len(f"{d.title}{DOC_SEP}".lstrip()) if d.title else 0   # some titles have leading spaces that strip() removes
    st, en = locate_sentences(doc_text, sents, start=min(body_offset, len(doc_text)))
    rows.append({"doc_id": d.doc_id, "title": d.title, "text": d.text, "doc_text": doc_text,
                 "sentences": sents, "sent_start": st, "sent_end": en,
                 "n_sentences": len(sents), "n_chars": len(doc_text), "n_words": len(doc_text.split()),
                 "structured": bool(o.get("structured", False))})
corpus = pd.DataFrame(rows)
CORPUS_IDX = {d: i for i, d in enumerate(corpus.doc_id)}

n_unlocated = int(sum(sum(s < 0 for s in st) for st in corpus.sent_start))
n_sent_total = int(corpus.n_sentences.sum())
check("all abstract sentences located in doc_text", n_unlocated == 0, f"{n_unlocated}/{n_sent_total} not located")
check("every doc has sentence segmentation", (corpus.n_sentences > 0).all(), f"{(corpus.n_sentences == 0).sum()} docs without sentences")
corpus[["doc_id", "title", "n_sentences", "n_words", "structured"]].head()

### 2.5 Queries and qrels tables

In [ ]:
qrels = beir_qrels[["query_id", "doc_id", "relevance", "split"]].copy()
check("every qrel doc exists in corpus", qrels.doc_id.isin(CORPUS_IDX).all(),
      f"{(~qrels.doc_id.isin(CORPUS_IDX)).sum()} missing", hard=True)
check("no duplicate qrel rows", not qrels.duplicated(["query_id", "doc_id", "split"]).any(), hard=True)
q_split = qrels.drop_duplicates("query_id").set_index("query_id")["split"]
check("no query appears in both train and test qrels", qrels.groupby("query_id").split.nunique().max() == 1, hard=True)

queries = beir_queries.copy()
queries["split"] = queries.query_id.map(q_split).fillna("unused")
queries["orig_split"] = queries.query_id.map(lambda q: all_orig[q][0])
display(pd.crosstab(queries.split, queries.orig_split, margins=True))
print("'unused' = BEIR queries with no qrels (not evaluated, following the BEIR convention)")

### 2.6 Evidence mapping

For every qrel pair (claim, relevant document), find the original annotation `claims_<orig_split>.jsonl → evidence[doc_id]`. That is a list of **rationale sets**, each a list of sentence indices plus a label. We keep:

- `evidence_sentence_ids`: the union of all rationale sets (sorted), with `ev_char_start/ev_char_end` as the matching spans in `doc_text`
- `label`: SUPPORT / CONTRADICT (`""` if the pair has no rationale)
- one row per evidence sentence in `evidence_sentences` (keeps the rationale-set structure for anyone who needs it)

In [ ]:
pair_rows, sent_rows = [], []
for r in qrels.itertuples(index=False):
    osplit = SPLIT_MAP[r.split]
    claim = orig_claims[osplit].get(r.query_id, {})
    ev_sets = (claim.get("evidence") or {}).get(r.doc_id, [])
    doc = corpus.iloc[CORPUS_IDX[r.doc_id]]
    labels = sorted({e["label"] for e in ev_sets})
    sent_ids = sorted({int(i) for e in ev_sets for i in e["sentences"]})
    in_range = [i for i in sent_ids if 0 <= i < doc.n_sentences]
    starts = [int(doc.sent_start[i]) for i in in_range]
    ends = [int(doc.sent_end[i]) for i in in_range]
    located = [s for s in starts if s >= 0]
    pair_rows.append({
        "query_id": r.query_id, "doc_id": r.doc_id, "split": r.split, "orig_split": osplit,
        "relevance": int(r.relevance), "claim": claim.get("claim", ""),
        "has_evidence": bool(sent_ids),
        "label": labels[0] if len(labels) == 1 else ("|".join(labels) if labels else ""),
        "n_rationale_sets": len(ev_sets),
        "evidence_sentence_ids": sent_ids, "n_evidence_sentences": len(sent_ids),
        "n_doc_sentences": int(doc.n_sentences), "doc_n_chars": int(doc.n_chars),
        "ev_char_start": starts, "ev_char_end": ends,
        "spans_ok": bool(sent_ids) and len(in_range) == len(sent_ids) and len(located) == len(starts),
        "first_evidence_sentence": min(sent_ids) if sent_ids else -1,
        "last_evidence_sentence": max(sent_ids) if sent_ids else -1,
        "first_evidence_char": min(located) if located else -1,
        "last_evidence_char": max(e for s, e in zip(starts, ends) if s >= 0) if located else -1,
        "rel_position_first": (min(sent_ids) / doc.n_sentences) if sent_ids and doc.n_sentences else np.nan,
    })
    for k, e in enumerate(ev_sets):
        for i in e["sentences"]:
            i = int(i)
            ok = 0 <= i < doc.n_sentences
            sent_rows.append({
                "query_id": r.query_id, "doc_id": r.doc_id, "split": r.split, "label": e["label"],
                "rationale_set": k, "sentence_idx": i,
                "char_start": int(doc.sent_start[i]) if ok else -1,
                "char_end": int(doc.sent_end[i]) if ok else -1,
                "sentence_text": doc.sentences[i] if ok else "",
            })

qrel_pairs = pd.DataFrame(pair_rows)
evidence_sentences = pd.DataFrame(sent_rows)

# evidence annotations in the mapped original splits that do NOT appear as BEIR qrels
qrel_keys = set(zip(qrels.query_id, qrels.doc_id))
ev_not_in_qrels = [(qid, did) for bs, os_ in SPLIT_MAP.items() for qid, c in orig_claims[os_].items()
                   for did in (c.get("evidence") or {}) if (qid, did) not in qrel_keys]
qrel_pairs.head(3)

### 2.7 Mapping validation  *(Fair evaluation protocol, step 1: validate the rationale mapping before using it)*

In [ ]:
ev = qrel_pairs[qrel_pairs.has_evidence]
bad_range = sum(any(i >= n for i in ids) for ids, n in zip(ev.evidence_sentence_ids, ev.n_doc_sentences))
check("evidence sentence indices are within the abstract", bad_range == 0, f"{bad_range} pairs out of range", hard=True)
check("all evidence spans located in doc_text", ev.spans_ok.all(), f"{(~ev.spans_ok).sum()} pairs with unlocated spans")

# span text must reproduce the annotated sentence exactly (up to whitespace)
loc = evidence_sentences[evidence_sentences.char_start >= 0]
span_ok = [norm_ws(corpus.doc_text.iat[CORPUS_IDX[d]][s:e]) == norm_ws(t)
           for d, s, e, t in zip(loc.doc_id, loc.char_start, loc.char_end, loc.sentence_text)]
check("doc_text[char_start:char_end] == evidence sentence", all(span_ok), f"{len(span_ok) - sum(span_ok)} mismatches", hard=True)

check("one label per (claim, doc) pair", not ev.label.str.contains(r"\|").any(),
      f"{ev.label.str.contains(r'[|]').sum()} pairs with mixed labels")
check("every evidence annotation (mapped splits) appears in BEIR qrels", len(ev_not_in_qrels) == 0,
      f"{len(ev_not_in_qrels)} annotated pairs not in qrels")
# qrel pairs without a rationale are cited docs of NOINFO claims (BEIR counts cited docs as relevant)
no_ev = qrel_pairs[~qrel_pairs.has_evidence]
cited = [int(d) in orig_claims[o][q].get("cited_doc_ids", []) for q, d, o in zip(no_ev.query_id, no_ev.doc_id, no_ev.orig_split)]
check("every rationale-free qrel pair is a cited doc of its claim", all(cited),
      f"{len(no_ev)} such pairs, {len(cited) - sum(cited)} not cited — kept for retrieval eval, excluded from the evidence diagnostic")

summary = (qrel_pairs.groupby("split")
           .agg(queries=("query_id", "nunique"), qrel_pairs=("doc_id", "size"),
                pairs_with_evidence=("has_evidence", "sum"),
                SUPPORT=("label", lambda s: (s == "SUPPORT").sum()),
                CONTRADICT=("label", lambda s: (s == "CONTRADICT").sum()),
                mean_evidence_sents=("n_evidence_sentences", lambda s: s[s > 0].mean()),
                mean_rationale_sets=("n_rationale_sets", lambda s: s[s > 0].mean()))
           .reindex(SPLITS))
summary["evidence_coverage"] = summary.pairs_with_evidence / summary.qrel_pairs
summary.round(3)

### 2.8 Descriptive statistics

In [ ]:
test_ev = qrel_pairs[(qrel_pairs.split == "test") & qrel_pairs.spans_ok]
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))

axes[0].hist(corpus.n_words, bins=50, color="#5b7553")
axes[0].set(title="Document length (title + abstract)", xlabel="whitespace words", ylabel="documents")
axes[0].axvline(corpus.n_words.median(), color="k", ls="--", lw=1, label=f"median {corpus.n_words.median():.0f}")
axes[0].legend()

axes[1].hist(test_ev.rel_position_first, bins=np.linspace(0, 1, 21), color="#8c6d46")
axes[1].set(title="Where evidence starts (test pairs)", xlabel="first evidence sentence / #sentences", ylabel="pairs")

axes[2].hist(test_ev.first_evidence_char, bins=40, color="#46618c")
axes[2].set(title="First evidence character offset (test pairs)", xlabel="char offset in doc_text", ylabel="pairs")
plt.tight_layout()
fig.savefig(FIG_DIR / "s2_corpus_and_evidence_position.png", dpi=150)
plt.show()

pd.DataFrame({
    "doc words": corpus.n_words.describe(),
    "doc sentences": corpus.n_sentences.describe(),
    "evidence sents / pair": qrel_pairs.loc[qrel_pairs.has_evidence, "n_evidence_sentences"].describe(),
    "first evidence sentence idx": qrel_pairs.loc[qrel_pairs.has_evidence, "first_evidence_sentence"].describe(),
}).round(2)

### 2.9 Save processed files + manifest

In [ ]:
outputs = {"corpus": corpus, "queries": queries, "qrels": qrels,
           "qrel_pairs": qrel_pairs, "evidence_sentences": evidence_sentences}
for name, df in outputs.items():
    df.to_parquet(PROCESSED_FILES[name], index=False)

manifest = {
    "created_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "data_source": DATA_SOURCE, "split_map_beir_to_original": SPLIT_MAP, "doc_sep": DOC_SEP,
    "counts": {"docs": len(corpus), "queries": len(queries),
               **{f"{s}_eval_queries": int((queries.split == s).sum()) for s in SPLITS},
               **{f"{s}_qrel_pairs": int((qrels.split == s).sum()) for s in SPLITS},
               **{f"{s}_pairs_with_evidence": int(qrel_pairs[(qrel_pairs.split == s)].has_evidence.sum()) for s in SPLITS},
               "evidence_sentence_rows": len(evidence_sentences),
               "annotated_pairs_not_in_qrels": len(ev_not_in_qrels)},
    "files": {n: {"path": str(PROCESSED_FILES[n].relative_to(ROOT)), "sha256": sha256_file(PROCESSED_FILES[n])} for n in outputs},
    "raw_sha256": {str(p.relative_to(ROOT)): sha256_file(p) for p in BEIR_FILES + ORIG_FILES},
    "checks": CHECK_LOG, "environment": ENV,
}
(PROC_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2))
_CORPUS_IDS = None   # refresh cached corpus ids used by validate_run()
print(json.dumps(manifest["counts"], indent=2))
n_warn = sum(c["status"] != "PASS" for c in CHECK_LOG)
print(f"\nSection 2 done — {len(CHECK_LOG)} checks, {n_warn} warnings. Outputs in {PROC_DIR.relative_to(ROOT)}/")

---
# Checkpoint A · Load shared data  *(start here if you are skipping Section 2)*

Run Section 1, then this cell. It loads everything Sections 3–10 need from `data/processed/`.

| Variable | What |
|---|---|
| `corpus` | DataFrame, one row per doc. **Keep this order**: score matrices use columns in `DOC_IDS` order |
| `DOC_IDS`, `DOC_TEXTS` | lists aligned with `corpus` (the canonical `doc_text`) |
| `Q[split]` | evaluation queries for `split` (`query_id`, `claim`) |
| `QRELS[split]` | `{query_id: {doc_id: relevance}}` |
| `qrel_pairs`, `evidence_sentences` | evidence mapping (diagnostic use; test evidence never used for tuning) |

In [ ]:
corpus, queries, qrels, qrel_pairs, evidence_sentences = load_processed(
    "corpus", "queries", "qrels", "qrel_pairs", "evidence_sentences")
DOC_IDS = corpus["doc_id"].tolist()
DOC_TEXTS = corpus["doc_text"].tolist()
CORPUS_IDX = {d: i for i, d in enumerate(DOC_IDS)}
Q = {s: eval_queries(s, queries) for s in SPLITS}
QRELS = {s: qrels_dict(s, qrels) for s in SPLITS}
print(f"{len(DOC_IDS)} docs | " + " | ".join(f"{s}: {len(Q[s])} queries, {sum(map(len, QRELS[s].values()))} qrels" for s in SPLITS))
run_status()

---
# 3 · BM25 baseline  *(Member 3)*

Lexical reference method: BM25Okapi over the full tokenised `doc_text` (no truncation).

**Inputs (from Checkpoint A / disk)**
- `DOC_IDS`, `DOC_TEXTS` (index these, in this order)
- `Q["train"]`, `Q["test"]` (`query_id`, `claim`)
- `QRELS[split]` for a quick self-check

**Required outputs**
- `save_run(run, "bm25", "train")` and `save_run(run, "bm25", "test")` (build `run` with `scores_to_run(score_matrix, Q[split].query_id, DOC_IDS)`; keep **raw** BM25 scores)
- `log_efficiency("bm25", indexing_time_s=..., mean_latency_ms=..., p95_latency_ms=..., n_queries=...)`

**Notes & checklist**
- [ ] Choose and document the tokeniser (lowercase? punctuation? stop-words? stemming?) and keep it identical for docs and queries
- [ ] Record BM25 parameters (`k1`, `b`) and keep the rank-bm25 defaults unless the team agrees otherwise
- [ ] Warm up before timing the latency; time per query, then report mean and p95
- [ ] Sanity check against the proposal's preliminary test result: Recall@10 ≈ 0.776, nDCG@10 ≈ 0.652

In [ ]:
# ===================== 3.1 Tokeniser + BM25 index (time the indexing) =====================
import pickle
from rank_bm25 import BM25Okapi
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from nltk.stem import PorterStemmer   # PorterStemmer needs no nltk data download

# Tokeniser choice. Pick it on TRAIN only (fair protocol), then freeze it here.
BM25_TOKENISER = dict(lowercase=True, pattern=r"[a-z0-9]+", stopwords="sklearn_english", stemmer="porter")
BM25_PARAMS = dict(k1=1.5, b=0.75, epsilon=0.25)   # rank-bm25 defaults, recorded explicitly

_TOKEN_RE = re.compile(BM25_TOKENISER["pattern"])
_STOP = frozenset(ENGLISH_STOP_WORDS) if BM25_TOKENISER["stopwords"] else frozenset()
_stemmer = PorterStemmer() if BM25_TOKENISER["stemmer"] == "porter" else None
_stem_cache = {}

def _stem(w):
    if _stemmer is None:
        return w
    s = _stem_cache.get(w)
    if s is None:
        s = _stem_cache[w] = _stemmer.stem(w)
    return s

def bm25_tokenise(text):
    """The ONE tokeniser for both documents and queries."""
    text = text.lower() if BM25_TOKENISER["lowercase"] else text
    return [_stem(w) for w in _TOKEN_RE.findall(text) if w not in _STOP]

t0 = time.perf_counter()
doc_tokens = [bm25_tokenise(t) for t in DOC_TEXTS]          # same order as DOC_IDS
bm25 = BM25Okapi(doc_tokens, **BM25_PARAMS)
BM25_INDEX_TIME_S = time.perf_counter() - t0

bm25_path = CACHE_DIR / "bm25_index.pkl"
with open(bm25_path, "wb") as f:
    pickle.dump(bm25, f)
BM25_STORAGE_BYTES = bm25_path.stat().st_size

check("BM25 index covers every doc", bm25.corpus_size == len(DOC_IDS), f"{bm25.corpus_size} vs {len(DOC_IDS)}", hard=True)
n_empty = sum(len(t) == 0 for t in doc_tokens)
check("no empty tokenised documents", n_empty == 0, f"{n_empty} empty docs")
print(f"indexed {bm25.corpus_size} docs in {BM25_INDEX_TIME_S:.2f}s | vocab {len(bm25.idf):,} | "
      f"avg doc len {bm25.avgdl:.1f} tokens | pickle {BM25_STORAGE_BYTES/1e6:.1f} MB")



In [ ]:
# ===================== 3.2 Score all queries -> score matrix (n_queries x n_docs) -> scores_to_run -> save_run =====================
def bm25_score_matrix(claims):
    return np.vstack([bm25.get_scores(bm25_tokenise(c)) for c in claims])   # (n_queries, n_docs), raw scores

def quick_metrics(run, qrels, k=K):
    """Temporary self-check only. Member 1's Section 8 is the official evaluator."""
    top = run[run["rank"] <= k].groupby("query_id")["doc_id"].apply(list).to_dict()
    rec, ndcg = [], []
    for qid, rel in qrels.items():
        ranked = top.get(qid, [])
        hits = [1.0 if d in rel else 0.0 for d in ranked]
        rec.append(sum(hits) / len(rel))
        dcg = sum(h / np.log2(i + 2) for i, h in enumerate(hits))
        idcg = sum(1 / np.log2(i + 2) for i in range(min(len(rel), k)))
        ndcg.append(dcg / idcg)
    return {f"recall@{k}": float(np.mean(rec)), f"ndcg@{k}": float(np.mean(ndcg)), "n_queries": len(rec)}

BM25_RESULTS = {}
for split in SPLITS:
    qdf = Q[split]
    S = bm25_score_matrix(qdf["claim"].tolist())
    n_zero_q = int((S.max(axis=1) <= 0).sum())
    check(f"bm25/{split}: every query matches at least one doc", n_zero_q == 0, f"{n_zero_q} queries with all-zero scores")
    run = scores_to_run(S, qdf["query_id"].tolist(), DOC_IDS)
    save_run(run, "bm25", split)
    BM25_RESULTS[split] = quick_metrics(run, QRELS[split])
    print(split, BM25_RESULTS[split])

# Sanity check against the proposal's preliminary result (test split). Small gaps usually mean a different tokeniser.
r = BM25_RESULTS["test"]
check("test Recall@10 close to proposal pilot (0.776)", abs(r["recall@10"] - 0.776) < 0.01, f"{r['recall@10']:.4f}")
check("test nDCG@10 close to proposal pilot (0.652)", abs(r["ndcg@10"] - 0.652) < 0.01, f"{r['ndcg@10']:.4f}")



In [ ]:
# ===================== 3.3 Warm query latency + log_efficiency('bm25', ...) =====================
# Timing boundary (agree the same one with Members 4 and 5): one query at a time,
# query tokenisation + scoring all docs + top-K selection. Excludes index building and file I/O.
def bm25_search(claim, k=K):
    s = bm25.get_scores(bm25_tokenise(claim))
    top = np.argpartition(-s, k - 1)[:k]
    return top[np.argsort(-s[top])]

lat_claims = Q["test"]["claim"].tolist()
for c in lat_claims[:20]:          # warm-up, not timed
    bm25_search(c)

N_REPEATS = 3
lat_ms = []
for _ in range(N_REPEATS):
    for c in lat_claims:
        t0 = time.perf_counter()
        bm25_search(c)
        lat_ms.append((time.perf_counter() - t0) * 1000)
lat_ms = np.array(lat_ms)

eff = log_efficiency(
    "bm25",
    indexing_time_s=round(BM25_INDEX_TIME_S, 4),
    mean_latency_ms=round(float(lat_ms.mean()), 4),
    p95_latency_ms=round(float(np.percentile(lat_ms, 95)), 4),
    n_queries=len(lat_claims), n_repeats=N_REPEATS,
    n_embeddings=0, storage_bytes=BM25_STORAGE_BYTES,
    timing_boundary="tokenise query + score all docs + top-10 select; single query; warm",
    tokeniser=BM25_TOKENISER, params=BM25_PARAMS,
)
print(json.dumps(eff, indent=2, default=str))



---
# 4 · Prefix MiniLM  *(Member 4)*

Semantic baseline: one embedding per document from `all-MiniLM-L6-v2`. The model sees only the first `MAX_SEQ_LEN` (256) word-pieces of `doc_text`, including [CLS]/[SEP]; everything after that is cut off.

**Inputs (from Checkpoint A / disk)**
- `DOC_IDS`, `DOC_TEXTS` (encode exactly these strings; do **not** re-join title/text yourself)
- `Q[split]` claims, `MODEL_NAME`, `MAX_SEQ_LEN`

**Required outputs**
- Cached doc embeddings: `cache/prefix_doc_emb.npy` (float32, shape 5183×384, rows in `DOC_IDS` order) + `cache/prefix_doc_ids.json`
- Cached query embeddings per split: `cache/query_emb__{split}.npy` (+ ids). Member 5 reuses these, so the query encoding is shared
- `save_run(run, "prefix_minilm", split)` for train and test (cosine scores)
- `log_efficiency("prefix_minilm", indexing_time_s=..., n_embeddings=..., storage_bytes=..., mean_latency_ms=..., p95_latency_ms=...)`

**Notes & checklist**
- [ ] `model.max_seq_length = MAX_SEQ_LEN` (assert it), frozen model, `normalize_embeddings=True` so dot product = cosine
- [ ] Load from the cache if the file exists; otherwise encode and save
- [ ] Latency = query encoding + cosine scoring over the cached matrix, measured after a warm-up
- [ ] Record device (CPU/GPU/MPS) and batch size

In [ ]:
# 4.1 Load model, set max_seq_length, encode/cache document embeddings (time it)
# TODO (Member 4)

In [ ]:
# 4.2 Encode/cache query embeddings for train and test
# TODO (Member 4)

In [ ]:
# 4.3 Cosine scores -> scores_to_run -> save_run('prefix_minilm', split)
# TODO (Member 4)

In [ ]:
# 4.4 Warm latency, storage size, embedding count -> log_efficiency('prefix_minilm', ...)
# TODO (Member 4)

---
# 5 · Windowed MiniLM  *(Member 5)*

Extension: split each `doc_text` into overlapping windows that each fit in MiniLM's limit, embed every window, and score a document by its **best window** (MaxP: S_window(q,d) = max over windows of cos(e_q, e_w)).

**Inputs (from Checkpoint A / disk)**
- `DOC_IDS`, `DOC_TEXTS`, `MODEL_NAME`, `MAX_SEQ_LEN`
- Query embeddings from Section 4 (`cache/query_emb__{split}.npy`); re-use them rather than re-encoding

**Required outputs**
- `data/processed/windows.parquet`: `doc_id, window_id, char_start, char_end, n_wordpieces` (**char spans in `doc_text`** are needed by Section 6.2)
- Cached window embeddings `cache/window_emb.npy` (+ row → (doc_id, window_id) map)
- `save_run(run, "window_minilm", split)` for train and test
- `log_efficiency("window_minilm", indexing_time_s=..., n_embeddings=..., storage_bytes=..., mean_latency_ms=..., p95_latency_ms=...)`

**Notes & checklist**
- [ ] Window size ≤ `MAX_SEQ_LEN - 2` content word-pieces; choose a stride (overlap) and record it in the config below
- [ ] Build windows from **tokenizer offsets** so the char spans are exact (`return_offsets_mapping=True`)
- [ ] Window construction must **not** use evidence locations or test labels
- [ ] Documents that fit in one window should get exactly one window (identical to the prefix embedding; good sanity check)
- [ ] Report embedding count and storage against prefix (5,183 embeddings)

In [ ]:
# 5.0 Window config (record final choice here)
# WINDOW_SIZE = ...   # content word-pieces per window (<= MAX_SEQ_LEN - 2)
# STRIDE = ...        # step between window starts
# TODO (Member 5)

In [ ]:
# 5.1 Build windows with tokenizer offsets -> windows.parquet
# TODO (Member 5)

In [ ]:
# 5.2 Encode/cache window embeddings (time it)
# TODO (Member 5)

In [ ]:
# 5.3 MaxP aggregation -> doc score matrix -> scores_to_run -> save_run('window_minilm', split)
# TODO (Member 5)

In [ ]:
# 5.4 Warm latency, storage, embedding count -> log_efficiency('window_minilm', ...)
# TODO (Member 5)

---
# 6 · Truncation and evidence-retention diagnostic  *(Members 2 + 5)*

*Can the model see the evidence?* For each test (claim, relevant doc) pair with annotated evidence, we check whether the evidence characters fall inside the part of `doc_text` that the prefix model actually encodes.

| Step (proposal) | Where | Owner |
|---|---|---|
| 1 · Tokenisation and truncation tracking: tokenizer offsets → retained char span | 6.1 | M2 ✅ |
| 2 · Evidence localisation: char spans of the expert evidence sentences | Section 2.6 | M2 ✅ |
| 3 · Retention classification: complete / partial / absent | 6.1 | M2 ✅ |
| Window coverage of evidence | 6.2 | M5 |
| 4 · Recovery by visibility group, group sizes, gains/regressions, diagnostic records | 6.3 | M2 + M5 |

> **Caveat (keep in the report):** MiniLM truncation is a hypothesis to audit. Missing evidence does not prove that truncation caused a ranking failure.

### 6.1 Prefix retention  *(Member 2)*

- Tokenise every `doc_text` with the MiniLM tokenizer (offsets on, no truncation). The model keeps `MAX_SEQ_LEN − 2 = 254` content word-pieces, so `retained_char_end` is the end offset of the 254th word-piece, or the full length if the document is shorter.
- For each pair: `retained_frac` = share of evidence characters before `retained_char_end`.
  **complete** = 1.0 · **partial** = between 0 and 1 · **absent** = 0. The calculation uses the union of all rationale sets. `any_set_complete` records whether at least one full rationale set is visible.
- Computed for both splits so the diagnostic is available everywhere, but only **test** is reported, and train is never used to pick window settings.

In [ ]:
from transformers import AutoTokenizer
TOKENIZER_NAME = os.environ.get("LIBBOT_TOKENIZER", MODEL_NAME)   # env override only for offline testing
tok = AutoTokenizer.from_pretrained(TOKENIZER_NAME, use_fast=True)
assert tok.is_fast, "need a fast tokenizer for offset mapping"
N_SPECIAL = tok.num_special_tokens_to_add(pair=False)
CONTENT_BUDGET = MAX_SEQ_LEN - N_SPECIAL
print(f"tokenizer={TOKENIZER_NAME} | special tokens={N_SPECIAL} | content budget={CONTENT_BUDGET} word-pieces")

corpus, qrel_pairs, evidence_sentences = load_processed("corpus", "qrel_pairs", "evidence_sentences")

t0 = time.perf_counter()
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    enc = tok(corpus.doc_text.tolist(), add_special_tokens=False, truncation=False,
              return_offsets_mapping=True, verbose=False)
n_wp, cut = [], []
for text, offs in zip(corpus.doc_text, enc["offset_mapping"]):
    n_wp.append(len(offs))
    cut.append(len(text) if len(offs) <= CONTENT_BUDGET else int(offs[CONTENT_BUDGET - 1][1]))
print(f"tokenised {len(corpus)} docs in {time.perf_counter() - t0:.1f}s")

doc_stats = pd.DataFrame({"doc_id": corpus.doc_id, "n_wordpieces": n_wp, "n_chars": corpus.n_chars,
                          "retained_char_end": cut})
doc_stats["truncated"] = doc_stats.n_wordpieces > CONTENT_BUDGET
doc_stats["retained_char_frac"] = doc_stats.retained_char_end / doc_stats.n_chars.clip(lower=1)
doc_stats["n_sentences_fully_retained"] = [sum(e <= c for e in ends if e >= 0)
                                           for ends, c in zip(corpus.sent_end, cut)]

# cross-check: our cutoff matches what the model really receives with truncation on
sample = corpus.sample(min(200, len(corpus)), random_state=SEED)
real_len = [len(x) for x in tok(sample.doc_text.tolist(), truncation=True, max_length=MAX_SEQ_LEN)["input_ids"]]
ours = [min(n, CONTENT_BUDGET) + N_SPECIAL for n in doc_stats.set_index("doc_id").loc[sample.doc_id, "n_wordpieces"]]
check("retained length matches tokenizer(truncation=True, max_length=MAX_SEQ_LEN)", real_len == ours, hard=True)
n_title_cut = sum(c < len(t) for c, t in zip(cut, corpus.title))
check("truncation never cuts before the title ends", n_title_cut == 0, f"{n_title_cut} docs cut inside the title")
doc_stats.describe().round(2)

In [ ]:
CUT = dict(zip(doc_stats.doc_id, doc_stats.retained_char_end))

def retention(starts, ends, cutoff):
    total = sum(e - s for s, e in zip(starts, ends))
    kept = sum(max(0, min(e, cutoff) - s) for s, e in zip(starts, ends))
    full = sum(e <= cutoff for e in ends)
    none = sum(s >= cutoff for s in starts)
    return total, kept, full, len(starts) - full - none, none

# last evidence char of each rationale set, per (claim, doc)
SET_END = (evidence_sentences.groupby(["query_id", "doc_id", "rationale_set"]).char_end.max()
           .groupby(level=[0, 1]).apply(list).to_dict())

ret_rows = []
for p in qrel_pairs[qrel_pairs.has_evidence & qrel_pairs.spans_ok].itertuples(index=False):
    c = CUT[p.doc_id]
    total, kept, full, part, none = retention(p.ev_char_start, p.ev_char_end, c)
    frac = kept / total if total else np.nan
    any_set_complete = any(e <= c for e in SET_END.get((p.query_id, p.doc_id), []))
    ret_rows.append({"query_id": p.query_id, "doc_id": p.doc_id, "split": p.split, "label": p.label,
                     "retained_char_end": c, "doc_n_chars": p.doc_n_chars,
                     "n_evidence_sentences": p.n_evidence_sentences,
                     "ev_sent_full": full, "ev_sent_partial": part, "ev_sent_absent": none,
                     "retained_frac": frac,
                     "visibility": "complete" if frac >= 1 else ("absent" if frac <= 0 else "partial"),
                     "any_set_complete": any_set_complete})
prefix_retention = pd.DataFrame(ret_rows)
prefix_retention["doc_truncated"] = prefix_retention.doc_id.map(dict(zip(doc_stats.doc_id, doc_stats.truncated)))

doc_stats.to_parquet(PROCESSED_FILES["doc_token_stats"], index=False)
prefix_retention.to_parquet(PROCESSED_FILES["prefix_retention"], index=False)

order = ["complete", "partial", "absent"]
tab = pd.crosstab(prefix_retention.split, prefix_retention.visibility).reindex(index=SPLITS, columns=order, fill_value=0)
tab_pct = tab.div(tab.sum(axis=1), axis=0).mul(100).round(1).add_suffix(" %")
print(f"Docs longer than the prefix window: {doc_stats.truncated.sum()} / {len(doc_stats)} ({doc_stats.truncated.mean():.1%})")
display(pd.concat([tab, tab_pct], axis=1))

t = prefix_retention[prefix_retention.split == "test"]
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
axes[0].hist(doc_stats.n_wordpieces, bins=60, color="#5b7553")
axes[0].axvline(CONTENT_BUDGET, color="crimson", ls="--", label=f"cutoff ({CONTENT_BUDGET} content word-pieces)")
axes[0].set(title="Document length in MiniLM word-pieces", xlabel="word-pieces", ylabel="documents"); axes[0].legend()
counts = t.visibility.value_counts().reindex(order, fill_value=0)
axes[1].bar(counts.index, counts.values, color=["#5b7553", "#c9a227", "#a23b3b"])
for i, v in enumerate(counts.values):
    axes[1].text(i, v, f"{v} ({v / max(len(t), 1):.0%})", ha="center", va="bottom")
axes[1].set(title="Test pairs: evidence visible to prefix MiniLM?", ylabel="(claim, doc) pairs")
plt.tight_layout(); fig.savefig(FIG_DIR / "s6_prefix_retention.png", dpi=150); plt.show()

### 6.2 Window coverage of evidence  *(Member 5)*

**Inputs:** `windows.parquet` (Section 5) and `prefix_retention` / `qrel_pairs` (above).
**Output:** add `window_full_cover` (some single window contains all the evidence chars) and `window_any_cover` to a copy of `prefix_retention` → `data/processed/window_retention.parquet`.

In [ ]:
# 6.2 TODO (Member 5): evidence coverage by windows

### 6.3 Recovery by visibility group, gains and regressions  *(Members 2 + 5, Week 3)*

**Inputs:** `prefix_retention` (test), runs `prefix_minilm__test`, `window_minilm__test` (+ BM25 and hybrids for context).
**Plan:**
1. For each test pair, compute rank of the relevant doc under each method → `hit@10`.
2. Table: hit@10 by visibility group (complete / partial / absent) × method, **with group sizes** (n per group).
3. Gains and regressions: pairs where windowed hits@10 and prefix misses (gain), and the reverse (regression). List claim, doc title and evidence sentence for inspection.
4. Save `results/metrics/diagnostic_records.parquet` (one row per test pair × method).

In [ ]:
# 6.3 TODO (Members 2 + 5): recovery by visibility group

In [ ]:
# 6.3 TODO (Members 2 + 5): gain / regression inspection + save diagnostic_records.parquet

---
# 7 · Hybrid fusion and α tuning  *(Member 6)*

Per-query min-max normalisation of each method's scores, then $S_{hybrid}^{(r)} = \alpha_r \hat S_r + (1-\alpha_r)\hat S_{BM25}$, $r \in \{prefix, window\}$. Tune $\alpha$ on **train only**, freeze, then apply once to test.

**Inputs (from Checkpoint A / disk)**
- Runs: `load_run("bm25", split)`, `load_run("prefix_minilm", split)`, `load_run("window_minilm", split)` (top-`RUN_DEPTH` raw scores)
- `ALPHA_GRID` (same grid for both hybrids), `QRELS["train"]`

**Required outputs**
- `results/frozen_alphas.json`: `{"hybrid_prefix": α, "hybrid_window": α, "selected_by": "train nDCG@10", "grid": ALPHA_GRID}`
- `results/metrics/alpha_sweep.parquet` (method, α, train nDCG@10, Recall@10)
- `save_run(run, "hybrid_prefix", split)` and `save_run(run, "hybrid_window", split)` for train and test

**Notes & checklist**
- [ ] Decide and document the candidate pool for normalisation (union of the two runs' top-`RUN_DEPTH`; a doc missing from one run gets normalised score 0)
- [ ] Constant-score queries (max = min) need a guard against divide-by-zero
- [ ] α = 0 must reproduce BM25 ranking and α = 1 the semantic ranking (sanity test)
- [ ] Never look at test while choosing α

In [ ]:
# 7.1 Per-query min-max normalisation + fusion function
# TODO (Member 6)

In [ ]:
# 7.2 α sweep on train -> alpha_sweep.parquet, pick best by train nDCG@10, write frozen_alphas.json
# TODO (Member 6)

In [ ]:
# 7.3 Apply frozen α to train + test -> save_run('hybrid_prefix'/'hybrid_window', split)
# TODO (Member 6)

---
# 8 · Final evaluation  *(Member 1)*

One evaluation script for all five configurations on the **300 untouched test queries**.

**Inputs (from Checkpoint A / disk)**
- `load_run(method, "test")` for every method in `METHODS`, `QRELS["test"]`
- `results/efficiency/efficiency_log.jsonl`

**Required outputs**
- `results/metrics/per_query__test.parquet`: `method, query_id, recall@10, ndcg@10` (Member 6 bootstraps from this)
- `results/metrics/summary__test.csv`: mean Recall@10, nDCG@10 per method
- `results/metrics/efficiency_summary.csv`: indexing time, embeddings, storage, mean/p95 latency per method

**Notes & checklist**
- [ ] Recall@10 = |relevant ∩ top-10| / |relevant|, averaged over queries
- [ ] nDCG@10 with binary gains (qrels are all 1), log2 discount, ideal DCG from the qrels
- [ ] Cross-check one method with `pytrec_eval` or BEIR's `EvaluateRetrieval` (should match to 1e-6)
- [ ] Every query in `QRELS["test"]` is scored (missing run → 0, never dropped)

In [ ]:
# 8.1 Metric functions (recall@k, ndcg@k) + cross-check
# TODO (Member 1)

In [ ]:
# 8.2 Evaluate all runs on test -> per_query__test.parquet, summary__test.csv
# TODO (Member 1)

In [ ]:
# 8.3 Efficiency summary from efficiency_log.jsonl
# TODO (Member 1)

---
# 9 · Statistical comparison, results table, charts  *(Member 6)*

Compare all methods, with uncertainty.

**Inputs (from Checkpoint A / disk)**
- `results/metrics/per_query__test.parquet`, `efficiency_summary.csv`, `diagnostic_records.parquet` (6.3)

**Required outputs**
- Paired bootstrap 95% CIs for metric **differences** (e.g. window − prefix, hybrid_window − hybrid_prefix, each vs BM25) → `results/metrics/bootstrap_ci.csv`
- Final results table (quality + efficiency) → `results/metrics/final_table.csv`
- Charts in `results/figures/` (metric bars with CIs, accuracy–latency trade-off, diagnostic by visibility group)

**Notes & checklist**
- [ ] Resample **queries** with replacement (same resample for both methods = paired), e.g. 10,000 resamples, `SEED`
- [ ] Report small subgroup sizes next to subgroup results

In [ ]:
# 9.1 Paired bootstrap CIs
# TODO (Member 6)

In [ ]:
# 9.2 Final results table
# TODO (Member 6)

In [ ]:
# 9.3 Charts
# TODO (Member 6)

---
# 10 · Demo export and reproducibility log  *(Member 6 / Member 1)*

Prepare what the Streamlit demo (`demo/app.py`) reads, and record what produced the results.

**Inputs (from Checkpoint A / disk)**
- Everything in `data/processed/` and `results/`

**Required outputs**
- Demo reads processed files + test runs directly (no extra export needed unless the demo requires live querying; then cache models/embeddings under `cache/`)
- `results/repro_log.json`: config snapshot, environment, file hashes of runs, check log

**Notes & checklist**
- [ ] Re-run the notebook top-to-bottom from a clean clone before submission
- [ ] Confirm the numbers in the report match `final_table.csv`

In [ ]:
# 10.1 Demo support (Member 6)
# TODO

In [ ]:
# 10.2 Reproducibility log (Member 1)
# TODO